In [1]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall

import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

2026-03-29 14:01:11.537276: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774792871.732606      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774792871.790625      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774792872.222929      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774792872.222971      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774792872.222973      24 computation_placer.cc:177] computation placer alr

In [3]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-new-train-test-val"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train.npy'))
y_train = np.load(os.path.join(INPUT_PATH, 'y_train.npy'))
X_val = np.load(os.path.join(INPUT_PATH, 'X_val.npy'))
y_val = np.load(os.path.join(INPUT_PATH, 'y_val.npy'))
X_test = np.load(os.path.join(INPUT_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(INPUT_PATH, 'y_test.npy'))

In [4]:
import numpy as np

X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train phải là 3D, hiện tại là {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val phải là 3D, hiện tại là {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test phải là 3D, hiện tại là {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train và y_train lệch số mẫu"
assert len(X_val) == len(y_val), "X_val và y_val lệch số mẫu"
assert len(X_test) == len(y_test), "X_test và y_test lệch số mẫu"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (36597, 10, 133) float32
y_val:   (36597,) int32
X_test:  (45552, 10, 133) float32
y_test:  (45552,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [33804  2793]
Test class distribution: [42337  3215]


In [5]:
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional

def create_model(n_lstm_units, dropout_rate):
    model = Sequential([
        Bidirectional(
            LSTM(n_lstm_units, return_sequences=True),
            input_shape=(X_train.shape[1], X_train.shape[2])
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Bidirectional(
            LSTM(max(n_lstm_units // 2, 8), return_sequences=False)
        ),
        Dropout(dropout_rate),
        BatchNormalization(),

        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(curve='ROC', name='auroc'),
            tf.keras.metrics.AUC(curve='PR', name='auprc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )
    return model

print("✅ Đã khởi tạo create_model")

✅ Đã khởi tạo create_model


In [6]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}

print("Trọng số lớp Train:", class_weights_dict)

Trọng số lớp Train: {0: 0.5407133742841034, 1: 6.64048833819242}


### FIXED HYPERPARAMETERS FOR ENSEMBLE

In [7]:
BEST_UNITS = 16
BEST_DROPOUT = 0.3
BEST_BATCH_SIZE = 512

N_MODELS = 5
SEEDS = [42, 52, 62, 72, 82]
EPOCHS = 30

print("Ensemble configuration:")
print(f"BEST_UNITS      = {BEST_UNITS}")
print(f"BEST_DROPOUT    = {BEST_DROPOUT}")
print(f"BEST_BATCH_SIZE = {BEST_BATCH_SIZE}")
print(f"N_MODELS        = {N_MODELS}")
print(f"SEEDS           = {SEEDS}")
print(f"EPOCHS          = {EPOCHS}")

Ensemble configuration:
BEST_UNITS      = 16
BEST_DROPOUT    = 0.3
BEST_BATCH_SIZE = 512
N_MODELS        = 5
SEEDS           = [42, 52, 62, 72, 82]
EPOCHS          = 30


### TRAIN ENSEMBLE MEMBERS

In [8]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

val_probs_list = []
test_probs_list = []
member_summaries = []

for member_idx, seed in enumerate(SEEDS, start=1):
    print("\n" + "="*70)
    print(f"TRAINING ENSEMBLE MEMBER {member_idx}/{N_MODELS} | seed={seed}")
    print("="*70)

    # Dọn session cũ để tránh đầy RAM/GPU
    K.clear_session()
    gc.collect()

    # Cố định random seed để mỗi member có randomness riêng nhưng reproducible
    tf.keras.utils.set_random_seed(seed)

    # Tạo model với hyperparameter đã chốt từ baseline
    model = create_model(
        n_lstm_units=BEST_UNITS,
        dropout_rate=BEST_DROPOUT
    )

    # Callback cho từng member
    early_stopping = EarlyStopping(
        monitor='val_auroc',
        mode='max',
        patience=6,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_auroc',
        mode='max',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )

    # Train 1 member
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BEST_BATCH_SIZE,
        class_weight=class_weights_dict,
        shuffle=True,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

    # Lưu xác suất dự đoán để sau đó ensemble bằng average probability
    val_prob = model.predict(X_val, verbose=0).ravel()
    test_prob = model.predict(X_test, verbose=0).ravel()

    val_probs_list.append(val_prob)
    test_probs_list.append(test_prob)

    # Tìm epoch tốt nhất theo val_auroc
    if "val_auroc" in history.history and len(history.history["val_auroc"]) > 0:
        best_epoch = int(np.argmax(history.history["val_auroc"]) + 1)
        best_idx = best_epoch - 1
        best_val_auroc = float(np.max(history.history["val_auroc"]))
    else:
        best_epoch = None
        best_idx = -1
        best_val_auroc = np.nan

    # Lấy metric validation trực tiếp từ history tại best epoch
    member_summary = {
        "member": member_idx,
        "seed": seed,
        "best_epoch": best_epoch,
        "val_loss": history.history["val_loss"][best_idx] if "val_loss" in history.history and best_idx >= 0 else np.nan,
        "val_accuracy": history.history["val_accuracy"][best_idx] if "val_accuracy" in history.history and best_idx >= 0 else np.nan,
        "val_auroc": history.history["val_auroc"][best_idx] if "val_auroc" in history.history and best_idx >= 0 else np.nan,
        "val_auprc": history.history["val_auprc"][best_idx] if "val_auprc" in history.history and best_idx >= 0 else np.nan,
        "val_recall": history.history["val_recall"][best_idx] if "val_recall" in history.history and best_idx >= 0 else np.nan,
        "val_precision": history.history["val_precision"][best_idx] if "val_precision" in history.history and best_idx >= 0 else np.nan,
        "best_val_auroc_in_history": best_val_auroc
    }

    member_summaries.append(member_summary)

    print("\nValidation metrics of this member (from history at best epoch):")
    for key in ["val_loss", "val_accuracy", "val_auroc", "val_auprc", "val_recall", "val_precision"]:
        if key in history.history and best_idx >= 0:
            print(f"{key}: {history.history[key][best_idx]:.4f}")

    # Nếu muốn lưu từng model ra file thì bỏ comment dòng dưới
    # model.save(f"ensemble_member_{member_idx}.keras")

    # Giải phóng bớt bộ nhớ
    del model, history, val_prob, test_prob
    gc.collect()

# Chuyển thành mảng để cell sau dùng luôn
val_probs_array = np.vstack(val_probs_list)    # shape: (N_MODELS, len(X_val))
test_probs_array = np.vstack(test_probs_list)  # shape: (N_MODELS, len(X_test))

df_members = pd.DataFrame(member_summaries)

print("\n" + "="*70)
print("ENSEMBLE TRAINING FINISHED")
print("="*70)
print("val_probs_array shape :", val_probs_array.shape)
print("test_probs_array shape:", test_probs_array.shape)
print("\nMember summary:")
print(df_members)


TRAINING ENSEMBLE MEMBER 1/5 | seed=42


I0000 00:00:1774792908.846689      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774792908.853040      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1774792917.193527      76 cuda_dnn.cc:529] Loaded cuDNN version 91002


285/285 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.5239 - auprc: 0.1765 - auroc: 0.6952 - loss: 0.6438 - precision: 0.1178 - recall: 0.7610 - val_accuracy: 0.8046 - val_auprc: 0.3060 - val_auroc: 0.8380 - val_loss: 0.4465 - val_precision: 0.2394 - val_recall: 0.7168 - learning_rate: 0.0010
Epoch 2/30
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7734 - auprc: 0.3108 - auroc: 0.8520 - loss: 0.4845 - precision: 0.2253 - recall: 0.8099 - val_accuracy: 0.7964 - val_auprc: 0.3288 - val_auroc: 0.8432 - val_loss: 0.4303 - val_precision: 0.2358 - val_recall: 0.7440 - learning_rate: 0.0010
Epoch 3/30
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7966 - auprc: 0.3789 - auroc: 0.8859 - loss: 0.4266 - precision: 0.2529 - recall: 0.8548 - val_accuracy: 0.8262 - val_auprc: 0.3388 - val_auroc: 0.8422 - val_loss: 0.3753 - val_precision: 0.2598 - val_recall: 0.6910 - learning_rate: 0.0010
Epoch 4/30
285/285 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8112 - auprc: 0.4176 -

### AVERAGE PROBABILITIES OF ENSEMBLE MEMBERS

In [9]:

import numpy as np

ensemble_val_prob = np.mean(val_probs_array, axis=0)
ensemble_test_prob = np.mean(test_probs_array, axis=0)

print("ensemble_val_prob shape :", ensemble_val_prob.shape)
print("ensemble_test_prob shape:", ensemble_test_prob.shape)

print("\nValidation probability range:")
print("min =", ensemble_val_prob.min(), "| max =", ensemble_val_prob.max())

print("\nTest probability range:")
print("min =", ensemble_test_prob.min(), "| max =", ensemble_test_prob.max())

ensemble_val_prob shape : (36597,)
ensemble_test_prob shape: (45552,)

Validation probability range:
min = 0.009276817 | max = 0.9467446

Test probability range:
min = 0.009001938 | max = 0.9558054


### SCAN THRESHOLDS ON ENSEMBLE VALIDATION PROBABILITY

In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def eval_threshold(y_true, y_prob, th):
    y_pred = (y_prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    youden_j = sensitivity + specificity - 1

    return {
        "threshold": round(float(th), 2),
        "accuracy": acc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "youden_j": youden_j,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

rows = []
for th in np.arange(0.05, 0.96, 0.01):
    rows.append(eval_threshold(y_val, ensemble_val_prob, th))

df_th_ensemble = pd.DataFrame(rows)

print("Top 10 threshold theo Youden's J:")
print(df_th_ensemble.sort_values("youden_j", ascending=False).head(10))

print("\nTop 10 threshold theo F1:")
print(df_th_ensemble.sort_values("f1", ascending=False).head(10))

Top 10 threshold theo Youden's J:
    threshold  accuracy  sensitivity  specificity  precision    recall  \
34       0.39  0.756018     0.820265     0.750710   0.213753  0.820265   
37       0.42  0.770309     0.801647     0.767720   0.221881  0.801647   
41       0.46  0.788999     0.779449     0.789788   0.234515  0.779449   
33       0.38  0.750936     0.824203     0.744882   0.210690  0.824203   
32       0.37  0.745225     0.830648     0.738167   0.207681  0.830648   
42       0.47  0.793316     0.773720     0.794936   0.237655  0.773720   
35       0.40  0.760144     0.812388     0.755828   0.215623  0.812388   
45       0.50  0.806596     0.757250     0.810673   0.248385  0.757250   
44       0.49  0.802415     0.761905     0.805763   0.244767  0.761905   
36       0.41  0.765090     0.805943     0.761715   0.218416  0.805943   

          f1  youden_j     tn    fp   fn    tp  
34  0.339131  0.570975  25377  8427  502  2291  
37  0.347563  0.569367  25952  7852  554  2239  
41  

### CHOOSE FINAL THRESHOLD ON VALIDATION


In [11]:
# Rule: sensitivity >= 0.80, then maximize specificity

target_sensitivity = 0.80

candidate = df_th_ensemble[df_th_ensemble["sensitivity"] >= target_sensitivity].copy()

if len(candidate) == 0:
    print("Không có threshold nào đạt sensitivity mục tiêu.")
else:
    best_row = candidate.sort_values(
        ["specificity", "youden_j", "f1"],
        ascending=False
    ).iloc[0]

    best_threshold = float(best_row["threshold"])

    print("Best threshold for ensemble:")
    print(best_row)

    print(f"\nChosen best_threshold = {best_threshold:.2f}")

Best threshold for ensemble:
threshold          0.420000
accuracy           0.770309
sensitivity        0.801647
specificity        0.767720
precision          0.221881
recall             0.801647
f1                 0.347563
youden_j           0.569367
tn             25952.000000
fp              7852.000000
fn               554.000000
tp              2239.000000
Name: 37, dtype: float64

Chosen best_threshold = 0.42


### FINAL TEST EVALUATION FOR ENSEMBLE

In [12]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# ===== 1. Threshold-independent metrics =====
test_auroc = roc_auc_score(y_test, ensemble_test_prob)
test_auprc = average_precision_score(y_test, ensemble_test_prob)

# ===== 2. Threshold-dependent prediction =====
test_pred = (ensemble_test_prob >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)
acc = accuracy_score(y_test, test_pred)
youden_j = sensitivity + specificity - 1

print(f"=== ENSEMBLE TEST RESULT WITH THRESHOLD = {best_threshold:.2f} ===")

print("\n--- Threshold-independent metrics ---")
print(f"AUROC:      {test_auroc:.4f}")
print(f"AUPRC:      {test_auprc:.4f}")

print("\n--- Threshold-dependent metrics ---")
print(f"Accuracy:    {acc:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Youden's J:  {youden_j:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4))

=== ENSEMBLE TEST RESULT WITH THRESHOLD = 0.42 ===

--- Threshold-independent metrics ---
AUROC:      0.8792
AUPRC:      0.3726

--- Threshold-dependent metrics ---
Accuracy:    0.7813
Sensitivity: 0.8330
Specificity: 0.7774
Precision:   0.2213
Recall:      0.8330
F1-score:    0.3497
Youden's J:  0.6104

Confusion Matrix:
[[32912  9425]
 [  537  2678]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9839    0.7774    0.8686     42337
           1     0.2213    0.8330    0.3497      3215

    accuracy                         0.7813     45552
   macro avg     0.6026    0.8052    0.6091     45552
weighted avg     0.9301    0.7813    0.8319     45552

